# Run Gemma 4 12B IT with Authentication on Modal

This notebook deploys the Gemma 4 12B Instruct model and queries its secure web endpoint with proxy authentication.

### Step 1: Deploy the Model

In [ ]:
!uv run modal deploy ../llm-hosting/gemma-4-12B-it

### Step 2: Load Credentials and Setup Endpoint

In [1]:
import os
import requests

def load_dotenv():
    for path in [".env", "../.env", "../../.env"]:
        if os.path.exists(path):
            with open(path) as f:
                for line in f:
                    line = line.strip()
                    if line and not line.startswith("#") and "=" in line:
                        k, v = line.split("=", 1)
                        v = v.strip().strip("'\"")
                        os.environ.setdefault(k.strip(), v)
            break

load_dotenv()
MODAL_KEY = os.environ.get("MODAL_KEY")
MODAL_SECRET = os.environ.get("MODAL_SECRET")

if not MODAL_KEY or not MODAL_SECRET:
    print("WARNING: Credentials not loaded!")
else:
    print("Credentials loaded successfully!")

username = "sshibinthomass"
ENDPOINT_URL = f"https://{username}--gemma-4-12b-it-gemmamodel-generate-api.modal.run"
print(f"Target Endpoint URL: {ENDPOINT_URL}")

Credentials loaded successfully!
Target Endpoint URL: https://sshibinthomass--gemma-4-12b-it-gemmamodel-generate-api.modal.run


### Step 3: Run Authenticated Inference

In [6]:
def test_inference(
    prompt: str | list[dict],
    max_new_tokens: int = 256,
    temperature: float = 1.0,
    top_p: float = 0.95,
    top_k: int = 64,
    enable_thinking: bool = False
):
    headers = {
        "Content-Type": "application/json",
        "Modal-Key": MODAL_KEY,
        "Modal-Secret": MODAL_SECRET,
    }
    payload = {
        "prompt": prompt,
        "max_new_tokens": max_new_tokens,
        "temperature": temperature,
        "top_p": top_p,
        "top_k": top_k,
        "enable_thinking": enable_thinking
    }
    
    print("Sending request to Gemma model...")
    response = requests.post(ENDPOINT_URL, headers=headers, json=payload)
    if response.status_code == 200:
        return response.json()
    else:
        return f"Error {response.status_code}: {response.text}"

prompt = "Explain what serverless computing is in three simple sentences."
reply = test_inference(prompt)
print(f"\nResponse:\n{reply}")

Sending request to Gemma model...

Response:
Serverless computing is a cloud model where developers can run code without having to manage, provision, or maintain the underlying physical servers. Instead of renting a specific machine, the cloud provider automatically handles all the hardware and scaling based on how much traffic the application receives. You only pay for the exact amount of resources your code consumes while it is actually running.


### Step 4: Verify Security (Unauthorized Calls)

In [ ]:
print("--- Test: Querying WITHOUT credentials ---")
response = requests.post(ENDPOINT_URL, json={"prompt": "Test"})
print(f"Status: {response.status_code} (Expected: 401)")
print(f"Message: {response.text}")